In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.Utils import (
    CanonicalUnits, 
    GravitationalParameters, 
    trasformation_X_to_E, 
    compute_jacobian_XoE,
    trasformation_E_to_X
)
import numpy as np
import multimin as mm
from tqdm import tqdm

\begin{equation}
    p(\widetilde{X})|d\widetilde{X}| = \eta(\widetilde{\varepsilon})|d\widetilde{\varepsilon}|
\end{equation}

\begin{equation}
    p(\widetilde{X}) = \eta(\widetilde{\varepsilon})|\frac{d\widetilde{\varepsilon}}{d\widetilde{X}}|
\end{equation}

\begin{equation}
    p(\widetilde{X}) = \eta(\widetilde{\varepsilon}(\widetilde{X}))\text{ det }\mathbb{J}_{\varepsilon \widetilde{X}}
\end{equation}

\begin{equation}
    \mathbb{J}_{\varepsilon \widetilde{X}} \equiv \frac{\partial \widetilde{\varepsilon}}{\partial \widetilde{X}} \equiv 
    \begin{pmatrix} 
    \partial_xa & \partial_ya & \partial_za & \partial_{v_x}a  & \partial_{v_y}a & \partial_{v_z}a\\
    \partial_xe & \partial_ye & \partial_ze & \partial_{v_x} e & \partial_{v_y}e & \partial_{v_z}e \\
    \partial_xi & \partial_yi & \partial_zi & \partial_{v_x} i & \partial_{v_y}i & \partial_{v_z}i \\
    \partial_x \Omega & \partial_y \Omega & \partial_z \Omega & \partial_{v_x} \Omega & \partial_{v_y} \Omega & \partial_{v_z } \Omega \\
    \partial_xw & \partial_yw & \partial_zw & \partial_{v_x} w & \partial_{v_y}w & \partial_{v_z}w  \\
    \partial_xM & \partial_yM & \partial_zM & \partial_{v_x} M & \partial_{v_y}M & \partial_{v_z}M
    \end{pmatrix}
\end{equation}

In this case we have

\begin{equation}
    \mathbb{J}_{X \varepsilon } \equiv \frac{\partial \widetilde{X}}{\partial \widetilde{\varepsilon}} \equiv 
    \begin{pmatrix} 
    \partial_ax & \partial_ex & \partial_ix & \partial_\Omega x & \partial_wx & \partial_Mx\\
    \partial_ay & \partial_ey & \partial_iy & \partial_\Omega y & \partial_wy & \partial_My \\
    \partial_az & \partial_ez & \partial_iz & \partial_\Omega z & \partial_wz & \partial_Mz \\
    \partial_av_x & \partial_ev_x & \partial_iv_x & \partial_\Omega v_x & \partial_wv_x & \partial_Mv_x \\
    \partial_av_y & \partial_ev_y & \partial_iv_y & \partial_\Omega v_y & \partial_wv_y & \partial_Mv_y \\
    \partial_av_z & \partial_ev_z & \partial_iv_z & \partial_\Omega v_z & \partial_wv_z & \partial_Mv_z
    \end{pmatrix}
\end{equation}

So we can use this property: 

\begin{equation}
    \mathbb{J}_{\varepsilon \widetilde{X}} = \mathbb{J}_{X \varepsilon }^{-1}
\end{equation}

\begin{equation}
    \det \mathbb{J}_{\varepsilon \widetilde{X}} = \frac{1}{\det \mathbb{J}_{X \varepsilon}} 
\end{equation}

In [2]:
deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s
mu = CanonicalUnits().mu
grav_params = GravitationalParameters(mu=mu)

In [3]:
def P_E_CMND(a: float, e: float, i: float, F: mm.FitCMND) -> float:
    max = 2*np.pi; min = 0

    P_aei = F.cmnd.pdf([a, e, i])
    P_WwM = 1/(max - min)**3

    return P_aei * P_WwM

In [4]:
def P_X_CMND(x: np.array, y: np.array, z: np.array, vx: np.array, vy: np.array, vz: np.array, mu: float, F: mm.FitCMND) -> np.array:
    """
    Vectorized version: x, y, vx, vy are arrays (or scalars).
    Returns array of P values.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    z = np.asarray(z)
    vx = np.asarray(vx)
    vy = np.asarray(vy)
    vz = np.asarray(vz)
    # Prepare output array
    shape = np.broadcast(x, y, z, vx, vy, vz).shape
    P = np.empty(shape, dtype=float)

    # Flatten for iteration if needed
    x_flat = x.ravel()
    y_flat = y.ravel()
    z_flat = z.ravel()
    vx_flat = vx.ravel()
    vy_flat = vy.ravel()
    vz_flat = vz.ravel()

    problematic = []
    for idx in range(x_flat.size):
        a, e, i, Omega, w, M = trasformation_X_to_E(x_flat[idx], y_flat[idx], z_flat[idx], vx_flat[idx], vy_flat[idx], vz_flat[idx], mu)
        J = compute_jacobian_XoE(a,e,i,Omega,w,M,mu)
        with np.errstate(divide='ignore', invalid='ignore'):
            det = np.linalg.det(J)
            #print(x_flat[idx], y_flat[idx], z_flat[idx],  vx_flat[idx], vy_flat[idx], vz_flat[idx])
            #print(det)
            if det == 0 or not np.isfinite(det):
                problematic.append((x_flat[idx], y_flat[idx], z_flat[idx],  vx_flat[idx], vy_flat[idx], vz_flat[idx]))
                P.flat[idx] = np.nan
            else:
                inv_det = 1.0/det 
                print(P_E_CMND(a, e, i, F) * abs(inv_det)) 
                P.flat[idx] = P_E_CMND(a, e, i, F) * abs(inv_det)
    return P.reshape(shape), problematic

In [ ]:
N = int(1e7)
a_uniform = np.random.uniform(0, 2, N)
e_uniform = np.random.uniform(0, 1, N)
i_uniform = np.random.uniform(0, np.pi, N)
Omega_uniform = np.random.uniform(0, 2*np.pi, N)
w_uniform = np.random.uniform(0, 2*np.pi, N)
M_uniform = np.random.uniform(0, 2*np.pi, N)
q_uniform = a_uniform*(1-e_uniform)

elements = np.column_stack((a_uniform, e_uniform, i_uniform, Omega_uniform, w_uniform, M_uniform, q_uniform))

state_vectors = np.zeros((N, 6))
for el in tqdm(range(N)):
    a = a_uniform[el]
    e = e_uniform[el]
    i = i_uniform[el]
    Omega = Omega_uniform[el]
    w = w_uniform[el]
    M = M_uniform[el]

    x, y, z, vx, vy, vz = trasformation_E_to_X(a, e, i, Omega, w, M, mu)
    state_vectors[el] = np.array([x, y, z, vx, vy, vz])

In [ ]:

F = mm.FitCMND(f"../../multimin/products/fit-NEOS-Ngauss50.pkl")
for state_vector in state_vectors:
    x = state_vector[0]
    y = state_vector[1]
    z = state_vector[2]
    vx = state_vector[3]
    vy = state_vector[4]
    vz = state_vector[5]
    P_X = P_X_CMND(x, y, z, vx, vy, vz, mu, F)
    print(P_X)

In [5]:
def surface_integral_P_X_CMND(center, widths, n_points=8, mu=1, F=mm.FitCMND):
    """
    Calculate the surface integral of P_xyvxvy in a hypercube centered at (x, y, vx, vy)
    with dimensions (dx, dy, dvx, dvy) using Gauss-Legendre quadrature.

    Parameters:
        center: tuple/list/array of (x, y, vx, vy) center
        widths: tuple/list/array of (dx, dy, dvx, dvy) side lengths
        n_points: number of quadrature points per dimension

    Returns:
        Integral (float)
    """
    from numpy.polynomial.legendre import leggauss

    x0, y0, z0, vx0, vy0, vz0 = center
    dx, dy, dz, dvx, dvy, dvz = widths

    # Get Gauss-Legendre points and weights for [-1, 1]
    pts, wts = leggauss(n_points)

    # Map points from [-1, 1] to [center-width/2, center+width/2] for each dimension
    x_pts = x0 + 0.5*dx*pts
    y_pts = y0 + 0.5*dy*pts
    z_pts = z0 + 0.5*dz*pts
    vx_pts = vx0 + 0.5*dvx*pts
    vy_pts = vy0 + 0.5*dvy*pts
    vz_pts = vz0 + 0.5*dvz*pts

    # Create meshgrid of all quadrature points
    X, Y, Z, VX, VY, VZ = np.meshgrid(x_pts, y_pts, z_pts, vx_pts, vy_pts, vz_pts, indexing='ij')
    WX, WY, WZ, WVX, WVY, WVZ = np.meshgrid(wts, wts, wts, wts, wts, wts, indexing='ij')

    # Flatten for vectorized evaluation
    Xf = X.ravel()
    Yf = Y.ravel()
    Zf = Z.ravel()
    VXf = VX.ravel()
    VYf = VY.ravel()
    VZf = VZ.ravel()
    WF = (WX * WY * WZ * WVX * WVY * WVZ).ravel()
    # Evaluate P at all points
    Pf = P_X_CMND(Xf, Yf, Zf, VXf, VYf, VZf, mu, F=F)

    # Integral is sum(P * weight) * volume factor
    integral = np.sum(Pf * WF) * (0.5*dx) * (0.5*dy) * (0.5*dz) * (0.5*dvx) * (0.5*dvy) * (0.5*dvz)
    #return integral, y_points
    return integral

In [6]:
F = mm.FitCMND(f"../../multimin/products/fit-NEOS-Ngauss50.pkl")

# Define center and widths of the phase-space hypercube
c_x = 1
c_y = 0
c_z = 0
v_x = 0
v_y = (mu/1)**0.5
v_z = 0

dxyz = 0.2
dvxyz = 5000 * (1/AU_m) * year 

center = (c_x, c_y, c_z, v_x, v_y, v_z)
widths = (dxyz, dxyz, dxyz, dvxyz, dvxyz, dvxyz)

# Theoretical: compute expected number of objects in the volume by integrating the distribution
N_theoretical = surface_integral_P_X_CMND(center, widths, n_points=8, mu=mu, F=F)
print(f"Theoretical (integral) number of objects in volume: {N_theoretical}")

3.1084012332159854e-08
3.441332570971622e-08
4.0355496075419614e-08
4.7395646286662144e-08
5.1128178087487376e-08
4.866590824826667e-08
4.3540367301241126e-08
3.986718174788966e-08
3.3367254260407937e-08
3.686047116295178e-08
4.306294734465636e-08
5.035570210017029e-08
5.41941459671812e-08
5.164956335440675e-08
4.6333626623490546e-08
4.250343475185237e-08
3.7704271557600944e-08
4.1506549004467074e-08
4.819992594367142e-08
5.596985598802676e-08
6.000151156149763e-08
5.728040672725169e-08
5.158760792380313e-08
4.745350096484371e-08
4.407234526109995e-08
4.832423505551416e-08
5.5726085588534125e-08
6.417608099417087e-08
6.84640665642177e-08
6.545276472745912e-08
5.918951162133056e-08
5.460411989301056e-08
5.129350297325537e-08
5.606559791269037e-08
6.42710742681051e-08
7.347140404854974e-08
7.802836845992472e-08
7.468905323718399e-08
6.779330529370937e-08
6.270565842497108e-08
5.7020447927351434e-08
6.223687653668458e-08
7.111585249191542e-08
8.094120166457442e-08
8.57760006713284e-08
8.2

KeyboardInterrupt: 